In [1]:
import pandas as pd

web_data = pd.read_csv("cleaned_web_data.csv")
sessions_data = pd.read_csv("sessions_data.csv")

In [2]:
web_data["date_time"].head()

0    2017-04-15 13:00:34
1    2017-04-15 13:00:14
2    2017-04-15 12:58:35
3    2017-04-15 12:58:03
4    2017-04-15 12:57:56
Name: date_time, dtype: str

In [3]:
#Convert data type to datetime

web_data["date_time"] = pd.to_datetime(web_data["date_time"])
web_data["date_time"].head()

0   2017-04-15 13:00:34
1   2017-04-15 13:00:14
2   2017-04-15 12:58:35
3   2017-04-15 12:58:03
4   2017-04-15 12:57:56
Name: date_time, dtype: datetime64[us]

In [4]:
#Finding the start and end time of each session

session_time = web_data.groupby(["client_id", "visit_id"]).agg(
    start_time=("date_time", "min"),
    end_time=("date_time", "max")
).reset_index()

session_time.head()

,client_id,visit_id,start_time,end_time
0,555,637149525_38041617439_716659,2017-04-15 12:57:56,2017-04-15 13:00:34
1,647,40369564_40101682850_311847,2017-04-12 15:41:28,2017-04-12 15:47:45
2,934,7076463_57954418406_971348,2017-04-18 02:36:30,2017-04-18 02:38:52
3,1028,557292053_87239438319_391157,2017-04-08 18:51:28,2017-04-08 19:00:26
4,1104,543158812_46395476577_767725,2017-06-12 07:49:18,2017-06-12 07:49:18


In [5]:
# Calculating the total time in seconds

session_time["total_time_seconds"] = (
    session_time["end_time"] - session_time["start_time"]
).dt.total_seconds()

session_time.head()

,client_id,visit_id,start_time,end_time,total_time_seconds
0,555,637149525_38041617439_716659,2017-04-15 12:57:56,2017-04-15 13:00:34,158.0
1,647,40369564_40101682850_311847,2017-04-12 15:41:28,2017-04-12 15:47:45,377.0
2,934,7076463_57954418406_971348,2017-04-18 02:36:30,2017-04-18 02:38:52,142.0
3,1028,557292053_87239438319_391157,2017-04-08 18:51:28,2017-04-08 19:00:26,538.0
4,1104,543158812_46395476577_767725,2017-06-12 07:49:18,2017-06-12 07:49:18,0.0


In [6]:
# Adding the column to the sessions_data table

sessions_data = sessions_data.merge(
    session_time[["client_id", "visit_id", "total_time_seconds"]],
    on=["client_id", "visit_id"],
    how="left"
)

sessions_data.head(10)

,client_id,visit_id,step_count,total_time_seconds
0,555,637149525_38041617439_716659,5,158.0
1,647,40369564_40101682850_311847,5,377.0
2,934,7076463_57954418406_971348,4,142.0
3,1028,557292053_87239438319_391157,9,538.0
4,1104,543158812_46395476577_767725,1,0.0
5,1104,643221571_99977972121_69283,1,0.0
6,1186,507052512_11309370126_442139,1,0.0
7,1186,795373564_99931517312_810896,3,22.0
8,1195,393817425_39015278493_996341,5,245.0
9,1197,71862471_21202285428_848395,7,95.0


In [12]:
#Checking sessions with zero seconds

(sessions_data["total_time_seconds"] == 0).sum()

np.int64(11398)

In [14]:
sessions_data["total_time_seconds"].min()

np.float64(0.0)

In [15]:
sessions_data["total_time_seconds"].max()

np.float64(41965.0)

In [18]:
sessions_data["total_time_seconds"].nunique()

2661